In [3]:
!pip install -q datasets pandas numpy nltk

import pandas as pd
import numpy as np
import re
import nltk
from datasets import load_dataset

nltk.download("punkt")



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\ADITYA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\ADITYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADITYA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
# Load cleaned and stable CUAD clause classification dataset
dataset = load_dataset(
    "dvgodoy/CUAD_v1_Contract_Understanding_clause_classification"
)

dataset


C:\Users\ADITYA\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADITYA\.cache\huggingface\hub\datasets--dvgodoy--CUAD_v1_Contract_Understanding_clause_classification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnin

DatasetDict({
    train: Dataset({
        features: ['file_name', 'clause', 'pages', 'class_id', 'label', 'start_at', 'end_at'],
        num_rows: 13155
    })
})

In [5]:
# Convert training split to pandas DataFrame
df = pd.DataFrame(dataset["train"])

df.head()


,file_name,clause,pages,class_id,label,start_at,end_at
0,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,In the event that Licensor grants to another V...,2,8,Most Favored Nation,2558,2929
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"If Licensor enters, or has entered, into an ag...",8,8,Most Favored Nation,18515,19562
2,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,"Licensor shall provide to Rogers, no later tha...",8,8,Most Favored Nation,19563,20059
3,IntegrityMediaInc_20010329_10-K405_EX-10.17_23...,"If for any reason, Integrity and TL are subjec...",3,8,Most Favored Nation,8173,8345
4,TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4...,"The Company will, and Online BVI will cause th...",10,8,Most Favored Nation,30765,31577


In [6]:
print("Total number of clauses:", len(df))
print("\nTop clause categories:")
print(df["label"].value_counts().head(10))


Total number of clauses: 13155

Top clause categories:
label
Parties             2560
License Grant        721
Cap On Liability     636
Anti-Assignment      616
Audit Rights         615
Insurance            535
Document Name        521
Agreement Date       474
Expiration Date      456
Governing Law        455
Name: count, dtype: int64


In [7]:
def clean_text(text):
    """
    Cleans legal text while preserving semantic meaning.
    """
    text = text.lower()                       # lowercase
    text = re.sub(r"\n", " ", text)           # remove newlines
    text = re.sub(r"\s+", " ", text)          # normalize spaces
    text = re.sub(r"[^\w\s]", "", text)       # remove punctuation
    return text.strip()


In [9]:
df["clean_text"] = df["clause"].apply(clean_text)

df[["clause", "clean_text"]].head(3)


,clause,clean_text
0,In the event that Licensor grants to another V...,in the event that licensor grants to another v...
1,"If Licensor enters, or has entered, into an ag...",if licensor enters or has entered into an agre...
2,"Licensor shall provide to Rogers, no later tha...",licensor shall provide to rogers no later than...


In [11]:
from nltk.tokenize import sent_tokenize

# Ensure required NLTK tokenizer resource is available
nltk.download("punkt_tab")

df["sentences"] = df["clean_text"].apply(sent_tokenize)

df["sentences"].head(3)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ADITYA\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


0    [in the event that licensor grants to another ...
1    [if licensor enters or has entered into an agr...
2    [licensor shall provide to rogers no later tha...
Name: sentences, dtype: object

In [12]:
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

# Filter out very short clauses
df = df[df["word_count"] > 5]

print("Clauses after filtering:", len(df))


Clauses after filtering: 9699


In [13]:
df.sample(5)


,file_name,clause,pages,class_id,label,start_at,end_at,clean_text,sentences,word_count
12917,INTRICONCORP_03_10_2009-EX-10.22-Strategic All...,Cumulative annual HH & ALD Volume that use the...,10,19,Revenue/Profit Sharing,22693,22998,cumulative annual hh ald volume that use the ...,[cumulative annual hh ald volume that use the...,49
12523,"VIRTUALSCOPICS,INC_11_12_2010-EX-10.1-STRATEGI...","In the event that, during the Term of this Agr...",4,16,Rofr/Rofo/Rofn,9789,10308,in the event that during the term of this agre...,[in the event that during the term of this agr...,85
3205,WORLDWIDESTRATEGIESINC_11_02_2005-EX-10-RESELL...,Reseller grants to TouchStar a right and licen...,10,25,License Grant,42468,42592,reseller grants to touchstar a right and licen...,[reseller grants to touchstar a right and lice...,21
3309,EbixInc_20010515_10-Q_EX-10.3_4049767_EX-10.3_...,Subject to the terms and conditions of this Ag...,4,26,Non-Transferable License,18906,19313,subject to the terms and conditions of this ag...,[subject to the terms and conditions of this a...,60
11971,PlayboyEnterprisesInc_20090220_10-QA_EX-10.2_4...,Notwithstanding revenue actually generated by ...,17,21,Minimum Commitment,58992,59270,notwithstanding revenue actually generated by ...,[notwithstanding revenue actually generated by...,46


In [ ]:
import os

# Ensure processed directory exists
os.makedirs("../data/processed", exist_ok=True)
n
# Save preprocessed data
df.to_csv("../data/processed/cuad_preprocessed.csv", index=False)

print("Preprocessed dataset saved at data/processed/cuad_preprocessed.csv")


Preprocessed dataset saved at data/processed/cuad_preprocessed.csv
